# Lab 03 : Réseaux de neurones et deep learning

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atouammi/Formation_CNN_API_Docker_Module4/blob/main/Labs/03_neural_nets.ipynb)

Bienvenue dans ce nouveau lab de notre parcours en machine learning !
Dans le lab précédent, nous avons vu comment :

- Les modèles linéaires révèlent des tendances simples dans les données 📈
- La descente de gradient permet à un modèle d'apprendre en réduisant son erreur pas à pas 🏔️

Aujourd'hui, nous passons à l'outil le plus puissant de l'IA moderne :

### ✨ Les réseaux de neurones : des systèmes inspirés du cerveau, capables d'apprendre presque n'importe quoi.

### 🎯 Ce que vous allez apprendre aujourd'hui

1. **Les bases des réseaux de neurones** – comment des neurones artificiels apprennent à partir des données
2. **Construire votre premier réseau** – une démonstration pratique pas à pas
3. **Aller plus en profondeur** – pourquoi le deep learning (apprentissage « profond ») est si puissant
4. **Des applications réelles** – de la reconnaissance d'images à la traduction automatique
5. **Bonus** – construire le même réseau avec Keras et avec PyTorch

### 🧠 **L'idée centrale**

Les réseaux de neurones sont constitués de couches de petites unités de calcul – les neurones – qui :

- Reçoivent des entrées (par exemple des valeurs de pixels ou des caractéristiques d'un texte)
- Appliquent des poids et des fonctions d'activation non linéaires
- Transmettent leurs résultats vers l'avant pour produire une prédiction

Avec suffisamment de neurones et de couches, ces réseaux peuvent :

- Reconnaître des images complexes 🖼️
- Comprendre le langage et le contexte 🌍
- Apprendre des structures complexes qu'aucun humain ne pourrait programmer à la main

### 🔗 Dans la continuité des labs précédents

| Lab | Thème | Idée clé |
|-------|-------|----------|
| 02 Modèles linéaires | Ajuster des droites aux tendances | Le modèle suppose une structure simple |
| 03 Réseaux de neurones | Apprendre (presque) tout | Le modèle apprend lui-même la structure |
| 04 Réseaux convolutifs (CNN) | Classer des images médicales | Des filtres qui apprennent à repérer des motifs visuels |
| 05 Détection d'objets (YOLO) | Trouver des objets dans des images | Des réseaux profonds pré-entraînés, prêts à l'emploi |

Les réseaux de neurones **n'ont pas besoin de règles codées à la main**. Ils construisent leur propre compréhension en ajustant leurs paramètres internes à partir des données – ce qui en fait la base de tout, de ChatGPT aux voitures autonomes.

### 🌍 Pourquoi les réseaux de neurones sont importants

Ils font fonctionner de nombreux outils que vous utilisez au quotidien :

- 📸 La classification d'images sur votre téléphone
- 🧠 Les grands modèles de langage comme ChatGPT
- 🎧 Les recommandations de musique et de films
- 🚗 Les véhicules autonomes
- 🧬 Le diagnostic médical

Et ce n'est que le début.

---

Plongeons dans le sujet et construisons-en un de A à Z, puis voyons comment l'entraîner avec des bibliothèques utilisées en pratique comme TensorFlow/Keras et PyTorch.

Préparez-vous à entrer dans le monde du deep learning ! 🚀

In [1]:
# ⚙️ Configuration – exécutez cette cellule en premier.
# - Sur Google Colab : récupère les fichiers du cours (plotting_utils/, assets/, requirements.txt)
#   et installe ce qui manque.
# - En local (VS Code) : rien à faire, l'environnement est déjà installé avec requirements.txt.
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/Formation_CNN_API_Docker_Module4"
    if not os.path.exists(REPO_DIR):
        !git clone -q https://github.com/atouammi/Formation_CNN_API_Docker_Module4.git {REPO_DIR}
    %cd {REPO_DIR}/Labs

if "." not in sys.path:
    sys.path.append(".")
print("Environnement :", "Google Colab" if IN_COLAB else "local")

Environnement : local


In [2]:
# Note pédagogique : cette cellule importe les bibliothèques de réseaux de neurones et les fonctions de visualisation.

# Mise en place rapide : importer nos outils pour les réseaux de neurones

import numpy as np
import pandas as pd
from sklearn.datasets import make_circles, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers, Input

from plotting_utils.neural_nets import (
    plot_customer_data_scatter,
    create_interactive_neuron_designer,
    plot_circle_data_scatter,
    plot_neural_network_results,
    plot_digit_samples,
    create_interactive_digit_classifier,
)

In [3]:
# Note pédagogique : fixer la graine aléatoire rend les données synthétiques et l'initialisation du modèle plus faciles à comparer.

# Fixer la graine aléatoire pour des résultats reproductibles
np.random.seed(42)


## Partie 1 : Les réseaux de neurones, des cerveaux artificiels

### 🧠 **Comment fonctionne votre cerveau ?**

Votre cerveau compte environ 86 milliards de neurones qui :
1. **Reçoivent des signaux** d'autres neurones
2. **Traitent l'information** en combinant ces signaux
3. **Envoient un signal** à d'autres neurones s'ils sont activés
4. **Apprennent** en renforçant les connexions importantes

Les **réseaux de neurones artificiels** sont une *version simplifiée* de cette idée :

- Ils utilisent des nombres au lieu de signaux électriques.
- Ils « apprennent » en modifiant des poids grâce aux données et à un retour sur leurs erreurs.

### 🔗 **Du biologique à l'artificiel**

Voyons ce que fait un seul **neurone artificiel** :

| Composant | Description |
|-----------|-------------|
| **Entrées** | Des nombres qui représentent des caractéristiques (âge, revenu, etc.) |
| **Poids** | L'importance de chaque entrée (apprise pendant l'entraînement) |
| **Activation** | Décide s'il faut « s'activer » en fonction des entrées pondérées |
| **Sortie** | Un nombre transmis à la couche suivante |

Un **réseau de neurones** relie simplement beaucoup de ces neurones en couches :

- La **couche d'entrée** reçoit vos données
- Les **couches cachées** effectuent le traitement
- La **couche de sortie** donne la prédiction finale

### 🎯 Pourquoi un réseau fait mieux qu'un neurone seul

- Un neurone seul ne peut tracer qu'une ligne droite – c'est comme une régression linéaire.
- Un réseau avec plus de neurones et de couches peut :
  - Courber, plier et tordre les frontières 🌀
  - Combiner des motifs provenant de plusieurs entrées
  - Capturer des structures non linéaires dans les données

Développons notre intuition avec un exemple concret. 📊

### 🔍 Exemple : un seul neurone peut-il prédire qui va acheter ?

Nous allons utiliser un tout petit **jeu de données clients** avec seulement deux entrées : `age` (l'âge) et `income` (le revenu).

Chaque client a soit acheté un produit (1), soit non (0).

In [4]:
# Note pédagogique : ce jeu de données clients introduit la classification avec deux variables numériques faciles à comprendre.

customer_data = pd.DataFrame({
    'age': [25, 35, 45, 55, 30, 40, 50, 28, 38, 48],
    'income': [30, 50, 70, 90, 40, 60, 80, 35, 55, 75],
    'will_buy': [0, 0, 1, 1, 0, 1, 1, 0, 0, 1]
})
print(customer_data)

   age  income  will_buy
0   25      30         0
1   35      50         0
2   45      70         1
3   55      90         1
4   30      40         0
5   40      60         1
6   50      80         1
7   28      35         0
8   38      55         0
9   48      75         1


### 📈 Visualisons !

In [5]:
# Note pédagogique : le nuage de points permet de voir si les classes peuvent être séparées par une règle simple.

plot_customer_data_scatter(customer_data)

### 🧪 Tester un seul neurone

Simulons la façon dont un neurone artificiel traiterait un client. Supposons que ce client a 40 ans et un revenu de 60 000 $.

In [6]:
# Note pédagogique : cette fonction est un neurone artificiel : somme pondérée des entrées plus un biais, suivie d'un seuil.

def simple_neuron(age, income, weight_age=0.1, weight_income=0.05, bias=-3):
    weighted_sum = weight_age * age + weight_income * income + bias

    # On utilise une fonction d'activation sigmoïde
    activation = 1 / (1 + np.exp(-weighted_sum))
    prediction = 1 if activation > 0.5 else 0

    return prediction, activation

prediction, confidence = simple_neuron(age=40, income=60)
print(f"Prédiction : {'Va acheter' if prediction == 1 else 'Ne va pas acheter'}, confiance : {confidence:.2f}")

Prédiction : Va acheter, confiance : 0.98


### 🎮 Interactif : concevez votre propre neurone

Le code suivant utilise une fonction utilitaire pour visualiser comment un neurone artificiel fait ses prédictions.
Pas d'inquiétude : elle se contente de mettre en place les graphiques et l'interaction.

Utilisez maintenant les curseurs ci-dessous pour ajuster les poids et le biais, et observez les performances de votre neurone !
Trouverez-vous un réglage qui donne la plupart des prédictions justes ?

In [7]:
# Note pédagogique : cet outil interactif permet de régler les poids et le biais avant de passer à un vrai réseau de neurones.

create_interactive_neuron_designer(customer_data)

interactive(children=(FloatSlider(value=0.1, description='Weight (Age)', max=0.5, min=-0.5, step=0.01), FloatS…

### Ce que fait cet outil :

- Il montre à quel point vos poids sont efficaces.
- Le premier graphique montre les prédictions justes (en vert) et fausses (en rouge).
- Le second graphique montre la frontière de décision de votre neurone : la zone où il prédit « oui » et celle où il prédit « non ».

### Interpréter le résultat :

- Une frontière qui sépare bien les données signifie que votre neurone a appris quelque chose d'utile !
- Une frontière qui laisse passer beaucoup de points signifie que le neurone est trop simple, ou mal réglé.
- Regardez la précision (accuracy) affichée : visez au moins 80 % !

🧠 Cet exercice vous aide à comprendre intuitivement :
- Le rôle des poids et du biais
- Le fonctionnement des frontières de décision
- Pourquoi les vrais réseaux de neurones ont besoin de plus de neurones et de plus de couches

### 💡 À retenir

- Un neurone seul est utile, mais limité.
- Il a du mal avec les frontières courbes ou complexes.
- Vous venez de découvrir pourquoi on a besoin de réseaux de neurones multicouches pour traiter des structures complexes dans les données !

👉 Ensuite, nous allons passer d'un seul neurone à un vrai réseau, et voir à quel point les performances s'améliorent.

---

## Partie 2 : Construire votre premier réseau de neurones

### 🔄 D'un neurone seul à une machine qui apprend

Un neurone seul peut tracer une droite, mais il ne peut pas représenter des courbes, des angles ou des frontières de décision complexes.
Pour aller plus loin, il nous faut **beaucoup de neurones** qui travaillent ensemble, organisés en **couches**.

C'est l'idée centrale d'un **réseau de neurones**.

### 🧱 Que se passe-t-il à l'intérieur d'un réseau de neurones ?

- La **couche d'entrée** transmet les caractéristiques au réseau
- Les **couches cachées** traitent l'information étape par étape :
  - La première couche peut détecter des signaux simples
  - La suivante les combine en motifs
  - Les couches plus profondes construisent une compréhension plus abstraite
- La **couche de sortie** produit la prédiction finale

### 🌟 Pourquoi est-ce si puissant ?

- **Plusieurs neurones** permettent des frontières de décision souples et courbes
- **Plusieurs couches** permettent l'abstraction : des pixels → aux contours → aux formes → aux visages
- **Les fonctions d'activation** apportent au réseau sa « magie » non linéaire

Avec la bonne architecture, un réseau de neurones peut apprendre à peu près n'importe quoi.

Construisons-en un !


### 🌀 Étape 1 : créer un jeu de données difficile

Nous allons maintenant créer un jeu de données **impossible à séparer avec des droites**.
Il contient des motifs circulaires, que seul un réseau de neurones saura démêler.

In [8]:
# Note pédagogique : le jeu de données en cercles est volontairement non linéaire : une seule frontière droite ne suffit pas.

# Créer un jeu de données plus complexe, qui nécessite un réseau de neurones
X, y = make_circles(n_samples=300, noise=0.1, factor=0.3, random_state=42)

# Convertir en DataFrame pour faciliter la manipulation
circle_data = pd.DataFrame(X, columns=['x', 'y'])
circle_data['class'] = y

print("🎯 Jeu de données à motif complexe :")
print(f"   Nombre de points : {len(circle_data)}")
print(f"   Nombre de classes : {len(np.unique(y))}")

🎯 Jeu de données à motif complexe :
   Nombre de points : 300
   Nombre de classes : 2


### 🔍 Étape 2 : visualiser le problème

Traçons les données pour comprendre pourquoi c'est délicat.
Les deux classes forment des **cercles concentriques** : une seule droite ne peut pas les séparer.

In [9]:
# Note pédagogique : le graphique montre pourquoi les couches cachées sont utiles pour obtenir des frontières de décision courbes.

plot_circle_data_scatter(circle_data)

🧠 **Observation** :
- ❌ Les **modèles linéaires** vont échouer ici
- ❌ Un **neurone seul** n'y suffira pas
- ✅ Les **réseaux de neurones** sont à la hauteur du défi !


### 🛠️ Étape 3 : préparer les données

Nous allons maintenant séparer les données en jeux d'entraînement et de test, puis les mettre à l'échelle.
**Pourquoi mettre à l'échelle ?** Les réseaux de neurones sont sensibles à l'ordre de grandeur des valeurs d'entrée.

In [10]:
# Note pédagogique : la séparation et la mise à l'échelle préparent les données pour un entraînement Keras stable.

# Préparer les données
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Mettre les données à l'échelle (important pour les réseaux de neurones)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 🤖 Étape 4 : construire et entraîner un réseau de neurones

Créons un vrai réseau de neurones avec :
- **2 couches cachées**
- **10 neurones chacune**
- **Activation ReLU**
- **100 époques** d'entraînement (passages complets sur les données)

In [11]:
# Note pédagogique : ce modèle Keras utilise deux couches cachées pour apprendre une frontière de décision non linéaire.

# Construire un réseau de neurones simple avec Keras
model = keras.Sequential([
    Input(shape=(2,)),
    layers.Dense(10, activation='relu'),
    layers.Dense(10, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

# Compiler le modèle
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Entraîner le modèle et conserver l'historique d'entraînement
history = model.fit(X_train_scaled, y_train, epochs=100, verbose=0, validation_data=(X_test_scaled, y_test))

### 📈 Étape 5 : évaluer les performances

Testons maintenant la précision du réseau pour voir s'il a bien appris le motif.

In [12]:
# Note pédagogique : évaluer sur les jeux d'entraînement et de test permet de vérifier l'apprentissage et un éventuel surapprentissage.

# Évaluer sur les jeux d'entraînement et de test
train_loss, train_accuracy = model.evaluate(X_train_scaled, y_train, verbose=0)
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

# Afficher les résultats
print(f"\n🧠 Résultats du réseau de neurones :")
print(f"   Précision sur l'entraînement : {train_accuracy:.1%}")
print(f"   Précision sur le test : {test_accuracy:.1%}")
print(f"   Architecture du réseau : entrée → 10 → 10 → sortie")



🧠 Résultats du réseau de neurones :
   Précision sur l'entraînement : 99.5%
   Précision sur le test : 100.0%
   Architecture du réseau : entrée → 10 → 10 → sortie


🎉 Bravo ! Vous venez d'entraîner un vrai réseau de neurones sur un jeu de données qui mettrait en échec des modèles plus simples.

### 📊 Étape 6 : visualiser les performances du réseau de neurones

Maintenant que notre réseau est entraîné avec TensorFlow/Keras, voyons ce qu'il a réellement appris.

Nous allons regarder :
- La frontière de décision : où le modèle prédit une classe plutôt que l'autre
- Les courbes de perte (loss) : comment le modèle s'est amélioré pendant l'entraînement

In [13]:
# Note pédagogique : ce graphique relie la précision, les courbes de perte et la frontière de décision apprise.

plot_neural_network_results(X, y, model, scaler, history, test_accuracy)

### 💬 Que peut-on en conclure ?

- La **frontière de décision** montre que le réseau de neurones a appris à bien séparer les classes circulaires – ce qu'**un modèle linéaire ne pourrait jamais faire**.
- Les **courbes de perte** confirment que le modèle a réduit son erreur pendant l'entraînement et qu'il généralise bien à des données qu'il n'a jamais vues (la perte de validation reste basse).

Les réseaux de neurones **apprennent automatiquement des structures complexes** grâce à leurs multiples couches et à leurs activations non linéaires : c'est là toute leur force !

---

## 🎮 Explorer les réseaux de neurones visuellement (facultatif)

Envie de **jouer avec les neurones, les couches et les activations** sans écrire de code ?

👉 Rendez-vous sur le [**TensorFlow Playground**](https://playground.tensorflow.org/) !

Vous pourrez :
- Ajouter ou supprimer des couches cachées
- Essayer différentes fonctions d'activation
- Régler le taux d'apprentissage et la régularisation
- Observer comment le réseau s'adapte à des motifs complexes comme des **cercles concentriques**

🔍 **Astuce :** dans le Playground, choisissez le jeu de données à deux cercles (en haut à gauche).
C'est le même que celui sur lequel nous venons d'entraîner notre réseau !

Cet outil interactif est idéal pour **voir** ce qu'apprend un réseau de neurones, en temps réel.


---

## 🖼️ Le deep learning en action : reconnaître des chiffres manuscrits

Nous avons vu que les réseaux de neurones savent apprendre des formes courbes, mais savent-ils reconnaître des **images** ?

Dans cet exemple, nous allons utiliser de vraies données d'images :
- Chaque image ne fait que **8×8 pixels en niveaux de gris**
- Chacune représente un **chiffre manuscrit (de 0 à 9)**
- Votre objectif : construire un réseau de neurones profond qui **reconnaît les chiffres automatiquement**

C'est une **version miniature de la reconnaissance d'images**, qui montre comment le deep learning permet des tâches comme l'OCR (reconnaissance optique de caractères).

In [14]:
# Note pédagogique : le jeu de données de chiffres fait passer de données jouets à une vraie petite tâche de classification d'images.

digits = load_digits()
X_digits, y_digits = digits.data, digits.target

print("🖼️ Jeu de données de chiffres manuscrits :")
print(f"   Images : {len(X_digits)}")
print(f"   Pixels par image : {X_digits.shape[1]} (grille 8x8)")
print(f"   Classes : {len(set(y_digits))} chiffres (0–9)")

🖼️ Jeu de données de chiffres manuscrits :
   Images : 1797
   Pixels par image : 64 (grille 8x8)
   Classes : 10 chiffres (0–9)


### 👁️‍🗨️ Regardons les données

Chaque chiffre est représenté par une liste de 64 nombres (8×8 pixels en niveaux de gris). Visualisons-en quelques-uns.


In [15]:
# Note pédagogique : afficher des exemples de chiffres aide à comprendre les entrées de 8x8 pixels.

plot_digit_samples(X_digits, y_digits)

### 🧠 Étape 1 : préparer les données

On met les valeurs des pixels à l'échelle (comme pour les cercles) et on sépare les données en jeux d'entraînement et de test.


In [16]:
# Note pédagogique : les images aplaties sont séparées et mises à l'échelle avant d'être transmises au réseau de neurones.

# Séparer et mettre à l'échelle les données
X_train_digits, X_test_digits, y_train_digits, y_test_digits = train_test_split(
    X_digits, y_digits, test_size=0.3, random_state=42
)

scaler_digits = StandardScaler()
X_train_digits_scaled = scaler_digits.fit_transform(X_train_digits)
X_test_digits_scaled = scaler_digits.transform(X_test_digits)

### 🤖 Étape 2 : construire un réseau de neurones profond

Nous construisons maintenant un **réseau plus profond** qui :
- Apprend à partir des 64 pixels d'entrée
- Utilise des **activations ReLU**
- Possède **3 couches cachées** de taille décroissante (100 → 50 → 25)
- Prédit le bon chiffre (0–9) grâce à une **couche de sortie softmax**

In [17]:
# Note pédagogique : ce réseau associe 64 valeurs de pixels à 10 classes de chiffres grâce à une sortie softmax.

model_digits = keras.Sequential([
    layers.Input(shape=(64,)),  # images 8x8 aplaties en 64 pixels
    layers.Dense(100, activation='relu'),
    layers.Dense(50, activation='relu'),
    layers.Dense(25, activation='relu'),
    layers.Dense(10, activation='softmax')  # 10 classes en sortie
])

model_digits.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_digits = model_digits.fit(
    X_train_digits_scaled, y_train_digits,
    epochs=30, validation_data=(X_test_digits_scaled, y_test_digits),
    verbose=0
)

### 📈 Étape 3 : vérifier la précision

Voyons les performances du modèle sur les jeux d'entraînement et de test.


In [18]:
# Note pédagogique : l'évaluation mesure les performances du classifieur de chiffres sur des images de test jamais vues.

train_loss, train_acc = model_digits.evaluate(X_train_digits_scaled, y_train_digits, verbose=0)
test_loss, test_acc = model_digits.evaluate(X_test_digits_scaled, y_test_digits, verbose=0)

print(f"🎯 Résultats du deep learning :")
print(f"   Précision sur l'entraînement : {train_acc:.1%}")
print(f"   Précision sur le test : {test_acc:.1%}")

🎯 Résultats du deep learning :
   Précision sur l'entraînement : 100.0%
   Précision sur le test : 98.0%


### 🔍 Étape 4 : tester le réseau sur de nouveaux chiffres

Demandons au modèle de classer quelques chiffres qu'il n'a jamais vus, et regardons son niveau de confiance.

In [19]:
# Note pédagogique : des exemples de test tirés au hasard rendent les prédictions concrètes en comparant l'étiquette attendue et l'étiquette prédite.

# Choisir 5 chiffres au hasard dans le jeu de test
np.random.seed(42)
sample_indices = np.random.choice(len(X_test_digits), size=5, replace=False)

for idx in sample_indices:
    true_label = y_test_digits[idx]
    image = X_test_digits_scaled[idx].reshape(1, -1)
    prediction = model_digits.predict(image, verbose=0)
    predicted_label = np.argmax(prediction)
    confidence = np.max(prediction)

    print(f"🔢 Réel : {true_label} | Prédit : {predicted_label} | Confiance : {confidence:.1%}")

🔢 Réel : 2 | Prédit : 2 | Confiance : 100.0%
🔢 Réel : 4 | Prédit : 4 | Confiance : 99.9%
🔢 Réel : 0 | Prédit : 0 | Confiance : 99.9%
🔢 Réel : 6 | Prédit : 6 | Confiance : 100.0%
🔢 Réel : 2 | Prédit : 2 | Confiance : 100.0%


### ✅ Résumé : ce que vous avez appris

- Les réseaux de neurones vont **au-delà des motifs synthétiques** et savent traiter de vraies images.
- Avec suffisamment de profondeur et d'entraînement, ils classent les chiffres avec **plus de 95 % de précision** !
- C'est une version simplifiée de ce qui fait fonctionner de vrais outils comme l'OCR, la résolution de CAPTCHA ou le tri postal.

Pour aller plus loin :
- Découvrir comment les **couches convolutives** (CNN) améliorent la reconnaissance d'images
- Explorer des jeux de données plus complexes comme **MNIST** ou **CIFAR-10**


## 🎮 À vous de jouer : le classifieur de chiffres en action

Utilisez le curseur ci-dessous pour voir comment le réseau de neurones se comporte sur différents chiffres manuscrits.

Observez :
- Ce qu'il reconnaît **correctement** ✅
- Ce qu'il **classe mal** ❌
- À quel point il est **sûr** de ses réponses 🔍

In [20]:
# Note pédagogique : le classifieur interactif permet d'inspecter une à une les erreurs et les prédictions les plus sûres.

create_interactive_digit_classifier(X_test_digits, y_test_digits, X_test_digits_scaled, model_digits)

interactive(children=(IntSlider(value=0, continuous_update=False, description='Digit #:', max=539), Output()),…

## 🧠 Bonus : les réseaux de neurones avec PyTorch

Keras (TensorFlow) est idéal pour prototyper rapidement, mais PyTorch offre plus de flexibilité et est très utilisé en recherche.

Voici comment construire et entraîner le même réseau de neurones avec **PyTorch**.

Nous allons :
- Définir une classe de réseau de neurones personnalisée
- Utiliser la perte d'entropie croisée binaire et l'optimiseur Adam
- Entraîner le modèle et évaluer sa précision

In [21]:
# Note pédagogique : cette partie PyTorch reprend la démarche Keras pour montrer un autre framework de réseaux de neurones très répandu.

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score

# Convertir les données en tenseurs PyTorch
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

# Définir un réseau de neurones à propagation avant (feedforward) simple
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 10),
            nn.ReLU(),
            nn.Linear(10, 10),
            nn.ReLU(),
            nn.Linear(10, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# Instancier le modèle, définir la fonction de perte et l'optimiseur
model = SimpleNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Boucle d'entraînement
epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

# Évaluer sur les données de test
model.eval()
with torch.no_grad():
    test_preds = model(X_test_tensor).numpy()
    test_preds_class = (test_preds > 0.5).astype(int)

accuracy = accuracy_score(y_test, test_preds_class)

print(f"✅ Précision PyTorch sur le test : {accuracy:.1%}")

✅ Précision PyTorch sur le test : 100.0%


### 🔄 Quelles différences ?

- On définit sa propre classe `nn.Module` pour décrire l'architecture
- L'entraînement passe par une boucle écrite à la main (ce qui donne plus de contrôle)
- PyTorch fonctionne très bien avec **autograd** (dérivation automatique) et **l'accélération GPU**

Keras et PyTorch sont tous deux excellents : le choix dépend souvent de la tâche et de vos préférences.

🧪 Essayez de recréer ce réseau sur d'autres jeux de données !

## 🎓 Félicitations : vous avez terminé la partie « fondamentaux » du cours !

Au fil de ces labs, vous avez posé des bases solides en machine learning, des modèles simples jusqu'aux réseaux de neurones profonds qui font fonctionner l'IA moderne.

---

### 🧭 Votre parcours

**🔹 Lab 02 : modèles linéaires**
- Découvrir comment une machine apprend à partir de données avec des modèles en ligne droite
- Comprendre la descente de gradient comme outil d'optimisation
- Construire l'intuition mathématique derrière l'entraînement d'un modèle

**🔹 Lab 03 : réseaux de neurones et deep learning**
- Explorer comment des « neurones » organisés en couches apprennent des structures complexes et non linéaires
- Entraîner des modèles profonds à reconnaître des motifs dans des images et des données
- Voir comment l'IA fait fonctionner des outils modernes comme les assistants vocaux, la reconnaissance d'images et la traduction

**🔜 Lab 04 : réseaux convolutifs (CNN) et détection du cancer du sein**  
- Construire un CNN et utiliser le transfert d'apprentissage sur des échographies mammaires  

**🔜 Lab 05 : détection d'objets avec YOLO**
- Utiliser un réseau convolutif profond pré-entraîné pour détecter des objets dans une image

---

### 💡 Les grandes leçons

- 🧠 **Aucun modèle ne convient à tous les problèmes** : connaissez vos outils
- 🔍 **Les structures complexes demandent des modèles complexes**, mais aussi plus de données et plus de soin
- ⚖️ **Simplicité et performance sont toujours un compromis**
- 🔄 **Interpréter et valider les résultats est aussi important que construire le modèle**

---

### 🌐 Où le machine learning est présent autour de vous

- 🎬 **Les recommandations Netflix** : filtrage collaboratif + réseaux de neurones
- 📸 **La reconnaissance faciale** : réseaux convolutifs profonds
- ✉️ **La détection de spam** : arbres de décision sur des caractéristiques du texte
- 🚗 **La perception des voitures autonomes** : vision par ordinateur + apprentissage par renforcement
- 🌍 **La traduction en temps réel** : modèles de séquences et transformers

---

### 🚀 Et maintenant ?

**🔧 Pratique et projets**
- Appliquez ce que vous avez appris à des jeux de données de votre domaine
- Réalisez votre premier mini-projet de ML concret
- Utilisez des plateformes comme Kaggle ou Hugging Face pour explorer des jeux de données et des modèles

**📚 Continuer à apprendre**
- Approfondissez :
  - La vision par ordinateur
  - Le traitement automatique du langage (NLP)
  - L'apprentissage par renforcement
  - L'IA générative
- Explorez plus en profondeur TensorFlow, PyTorch et scikit-learn

💼 **Commencez à utiliser le machine learning dans votre domaine !**

---

### 🛠️ Le mot de la fin

Pas besoin d'un doctorat pour utiliser efficacement le machine learning.

Ce qui compte le plus :
- 🔍 **Savoir quel modèle convient au problème**
- 🧪 **Tester, valider et améliorer son modèle**
- 🧠 **Garder un regard critique sur les données et les biais**
- 📢 **Savoir expliquer ses résultats aux autres**

---

### 🌟 Vous êtes prêt

Ce n'est qu'un début : le ML est un voyage fait de curiosité, d'expérimentation et de créativité.

Explorez, construisez, posez des questions et créez quelque chose d'exceptionnel.
Nous avons hâte de voir ce que vous allez faire. 🚀🤖
